# ChessCNN — Lichess Supervised Pretraining

Behavioural cloning on Lichess games → baseline model for AlphaZero self-play on GCP.

**Runtime:** A100 GPU — *Runtime → Change runtime type → A100 GPU*

| Step | What happens | Est. time (A100) |
|------|-------------|------------------|
| Download | ~1-3 GB Lichess PGN | 5-10 min |
| Extract | Stream 1500+ ELO games, encode positions | 90-180 min |
| Train | up to 30 epochs + early stopping | 15-25 min |

**Dataset:** 1500+ ELO games → up to 100M positions, streamed in 2.5M-position chunks  
**GPU:** A100 80 GB — batch 32,768 + gradient checkpointing uses ~20 GB VRAM  
**Speedups:** `torch.compile` (~20%) + TF32 (~10%) + LR warmup for stability

After training: download `pretrained_model.pth` and upload to GCP as `game_engine/model/best_model.pth`.

In [ ]:
import subprocess, sys

r = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
print('GPU :', r.stdout.strip() if r.returncode == 0
      else 'NOT DETECTED — enable GPU in Runtime > Change runtime type')

print('Installing dependencies ...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'chess', 'zstandard', 'tqdm'], check=True)
print('Done.')

## Configuration
Edit everything here before running the rest of the notebook.

In [ ]:
import os

# ── Data source ───────────────────────────────────────────────────────────────
LICHESS_MONTH   = '2024-10'
MIN_TC_SECS     = 600         # 10-min+ time control
MIN_ELO         = 1500        # BOTH players must be >= this
CHUNK_POSITIONS = 2_500_000   # positions per chunk (not games)
MAX_CHUNKS      = 40          # stop after this many chunks (40 × 2.5M = 100M positions)
SKIP_HALF       = 8           # skip first N half-moves per game
SAMPLE_EVERY    = 3           # sample every Nth position after skip

# ── Training ──────────────────────────────────────────────────────────────────
EPOCHS         = 20
EARLY_STOP_PAT = 3
BATCH_SIZE     = 32_768
LR             = 8e-3
WEIGHT_DECAY   = 1e-4
VAL_SPLIT      = 0.05

# ── Parallel data loading ─────────────────────────────────────────────────────
MINI_CHUNK_SIZE = 32_768 * 32   # = 1,048,576 — 32 batches at 32,768 batch size
N_WORKERS       = 3              # 3 × 30.7 GB + 32 GB queue ≈ 124 GB < 130 GB RAM limit
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

# ── Storage ───────────────────────────────────────────────────────────────────
# USE_DRIVE = True  — mount Google Drive so everything persists across sessions.
#   On reconnect: re-run all cells. Download + extraction skipped automatically.
#   Training resumes from the last completed epoch automatically.
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/chess_pretrain'
else:
    BASE_DIR = '/content/chess_pretrain'

DATA_DIR    = os.path.join(BASE_DIR, 'data')
MODEL_PATH  = os.path.join(BASE_DIR, 'pretrained_model.pth')
RESUME_PATH = os.path.join(BASE_DIR, 'resume_checkpoint.pth')
os.makedirs(DATA_DIR, exist_ok=True)

print(f'Storage : {"Google Drive  (persists across sessions)" if USE_DRIVE else "local /content (ephemeral)"}')
print(f'Base    : {BASE_DIR}')
print(f'Resume  : {"found — will continue from last epoch" if os.path.exists(RESUME_PATH) else "not found — fresh run"}')


In [ ]:
print('=' * 60)
print('  ChessCNN Lichess Pretraining — Final Config')
print('=' * 60)
print(f'  Lichess month    : {LICHESS_MONTH}')
print(f'  Min time control : {MIN_TC_SECS}s  ({MIN_TC_SECS//60} min)')
print(f'  Min ELO (both)   : {MIN_ELO}+')
print(f'  Chunk size       : {CHUNK_POSITIONS:,} positions')
print(f'  Max chunks       : {MAX_CHUNKS}  ({MAX_CHUNKS * CHUNK_POSITIONS // 1_000_000:,}M positions target)')
print(f'  Skip half-moves  : {SKIP_HALF}  (opening book)')
print(f'  Sample every     : {SAMPLE_EVERY} half-moves')
print('-' * 60)
print(f'  Epochs           : {EPOCHS}  (hard ceiling)')
print(f'  Early stop       : patience {EARLY_STOP_PAT} epochs')
print(f'  Batch size       : {BATCH_SIZE:,}  (grad checkpoint → ~20 GB VRAM)')
print(f'  Learning rate    : {LR}  (with 1-epoch warmup)')
print(f'  Weight decay     : {WEIGHT_DECAY}')
print(f'  Val split        : {int(VAL_SPLIT*100)}%')
print(f'  Optimisations    : torch.compile + TF32 + bfloat16 AMP + grad checkpoint')
print('-' * 60)
_chunk_gb  = MINI_CHUNK_SIZE * 120 * 8 * 8 * 4 / 1e9
_queue_gb  = 32 * BATCH_SIZE * 120 * 8 * 8 * 4 / 1e9
_ram_est   = N_WORKERS * _chunk_gb + _queue_gb
print(f'  Mini-chunk size  : {MINI_CHUNK_SIZE:,} positions  ({_chunk_gb:.1f} GB decompressed)')
print(f'  Workers          : {N_WORKERS}  →  RAM ≈ {N_WORKERS}×{_chunk_gb:.1f} + {_queue_gb:.0f} = {_ram_est:.0f} GB')
print('=' * 60)


In [ ]:
import io, json, time, math, gc, types
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.checkpoint import checkpoint as grad_ckpt
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
import chess, chess.pgn, chess.engine
import zstandard as zstd
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import urllib.request
import psutil

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if torch.cuda.is_available():
    prop = torch.cuda.get_device_properties(0)
    print(f'GPU    : {prop.name}')
    print(f'VRAM   : {prop.total_memory / 1e9:.1f} GB')

## Model
Exact copy of `game_engine/cnn.py`. Keep in sync if the architecture changes.

In [ ]:
class Mish(nn.Module):
    def forward(self, x):
        return x * torch.tanh(F.softplus(x))

class SEBlock(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.squeeze    = nn.AdaptiveAvgPool2d(1)
        self.excitation = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False), Mish(),
            nn.Linear(channels // reduction, channels, bias=False), nn.Sigmoid()
        )
    def forward(self, x):
        b, c = x.size(0), x.size(1)
        y = self.squeeze(x).view(b, c)
        return x * self.excitation(y).view(b, c, 1, 1).expand_as(x)

class ResidualBlock(nn.Module):
    def __init__(self, channels, use_se=False):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(channels)
        self.act   = Mish()
        self.se    = SEBlock(channels) if use_se else None
    def forward(self, x):
        r   = x
        out = self.act(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.se: out = self.se(out)
        return self.act(out + r)

class ChessCNN(nn.Module):
    def __init__(self):
        super().__init__()
        filters, blocks = 256, 20
        self.input_conv = nn.Sequential(
            nn.Conv2d(120, filters, 3, padding=1, bias=False),
            nn.BatchNorm2d(filters), Mish()
        )
        self.res_blocks = nn.ModuleList(
            [ResidualBlock(filters, use_se=(i >= 10)) for i in range(blocks)]
        )
        self.policy_head = nn.Sequential(
            nn.Conv2d(filters, 32, 1, bias=False), nn.BatchNorm2d(32), Mish(),
            nn.Flatten(), nn.Linear(32 * 8 * 8, 4672)
        )
        self.value_head = nn.Sequential(
            nn.Conv2d(filters, 1, 1, bias=False), nn.BatchNorm2d(1), Mish(),
            nn.Flatten(), nn.Linear(64, 256), Mish(), nn.Linear(256, 3)
        )
    def forward(self, x):
        x = self.input_conv(x)
        for block in self.res_blocks:
            x = block(x)
        return self.policy_head(x), self.value_head(x)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True
torch.set_float32_matmul_precision('high')

model  = ChessCNN().to(device)
params = sum(p.numel() for p in model.parameters())
print(f'Parameters : {params:,}  ({params/1e6:.1f} M)')


def _fwd_checkpointed(self, x):
    x = self.input_conv(x)
    for block in self.res_blocks:
        x = grad_ckpt(block, x, use_reentrant=False)
    return self.policy_head(x), self.value_head(x)

model.forward = types.MethodType(_fwd_checkpointed, model)
print('Gradient checkpointing : enabled')

model = torch.compile(model)
print('torch.compile          : enabled')

# ── Warmup pass — triggers JIT compilation before training starts ────────────
# Without this, the first training batch is 10-30s slower due to graph compilation.
print('Running compile warmup ...')
_dummy = torch.randn(BATCH_SIZE, 120, 8, 8, device=device)
with torch.no_grad():
    _p, _v = model(_dummy)
print(f'Warmup done  |  Policy {tuple(_p.shape)}  Value {tuple(_v.shape)}')
del _dummy, _p, _v

## Encoders
Pure-Python equivalents of the C++ engine — board → 120-plane tensor, move → 4672 index.

In [ ]:
_PIECE_TYPES = [
    chess.PAWN, chess.KNIGHT, chess.BISHOP,
    chess.ROOK, chess.QUEEN,  chess.KING
]

def _fill_frame(tensor, board, offset):
    """14 planes at offset: 6 white pieces, 6 black pieces, 2 repetition."""
    for i, pt in enumerate(_PIECE_TYPES):
        for sq in board.pieces(pt, chess.WHITE):
            tensor[offset + i,     sq // 8, sq % 8] = 1.0
        for sq in board.pieces(pt, chess.BLACK):
            tensor[offset + 6 + i, sq // 8, sq % 8] = 1.0
    if board.is_repetition(2): tensor[offset + 12, :, :] = 1.0
    if board.is_repetition(3): tensor[offset + 13, :, :] = 1.0

def board_to_tensor(board, history=None):
    """
    (120, 8, 8) float32 — matches chess_board.cpp to_tensor() exactly.
    history: list[chess.Board] most-recent-first, up to 7 boards.
    """
    t = np.zeros((120, 8, 8), dtype=np.float32)
    for i, b in enumerate([board] + list(history or [])[:7]):
        _fill_frame(t, b, i * 14)
    if board.has_kingside_castling_rights(chess.WHITE):  t[112] = 1.0
    if board.has_queenside_castling_rights(chess.WHITE): t[113] = 1.0
    if board.has_kingside_castling_rights(chess.BLACK):  t[114] = 1.0
    if board.has_queenside_castling_rights(chess.BLACK): t[115] = 1.0
    if board.ep_square is not None:
        t[116, board.ep_square // 8, board.ep_square % 8] = 1.0
    if board.turn == chess.BLACK: t[117] = 1.0
    t[118] = board.fullmove_number / 512.0
    t[119] = board.halfmove_clock  / 100.0
    return t

_KNIGHT_MAP = {d: i for i, d in enumerate(
    [(1,2),(2,1),(2,-1),(1,-2),(-1,-2),(-2,-1),(-2,1),(-1,2)]
)}

def move_to_index(move):
    """
    AlphaZero 4672 encoding: src*73 + plane.
    Matches mcts_engine.h move_to_index() exactly.
    """
    src = move.from_square
    df  = chess.square_file(move.to_square) - chess.square_file(src)
    dr  = chess.square_rank(move.to_square) - chess.square_rank(src)
    if move.promotion and move.promotion != chess.QUEEN:
        piece = {chess.KNIGHT: 0, chess.BISHOP: 1, chess.ROOK: 2}[move.promotion]
        return src * 73 + 64 + (0 if df < 0 else 1 if df == 0 else 2) * 3 + piece
    adf, adr = abs(df), abs(dr)
    if (adf, adr) in ((1,2),(2,1)):
        return src * 73 + 56 + _KNIGHT_MAP[(df, dr)]
    if   df == 0:           d = 0 if dr>0 else 4;  dist = adr-1
    elif dr == 0:           d = 2 if df>0 else 6;  dist = adf-1
    elif df>0 and dr>0:     d = 1;                 dist = adf-1
    elif df<0 and dr>0:     d = 7;                 dist = adf-1
    elif df>0 and dr<0:     d = 3;                 dist = adf-1
    else:                   d = 5;                 dist = adf-1
    return src * 73 + d * 7 + dist

# Sanity checks
_b = chess.Board()
_t = board_to_tensor(_b)
assert _t.shape == (120, 8, 8)
assert _t[0, 1, :].sum() == 8
assert move_to_index(chess.Move.from_uci('e2e4')) == chess.E2 * 73 + 0 * 7 + 1
assert move_to_index(chess.Move.from_uci('g1f3')) == chess.G1 * 73 + 56 + _KNIGHT_MAP[(-1,2)]
print('board_to_tensor  OK')
print('move_to_index    OK')

## Data Pipeline
Downloads one Lichess month, streams games where both players are 1500+ ELO, encodes positions in 2.5M-position chunks, and splits into train/val.

In [ ]:
def download_lichess(month, dest_dir):
    url  = (f'https://database.lichess.org/standard/'
            f'lichess_db_standard_rated_{month}.pgn.zst')
    path = os.path.join(dest_dir, f'lichess_{month}.pgn.zst')
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f'Already downloaded  ({size_mb:.0f} MB) : {path}')
        return path

    print(f'Downloading: {url}')
    bar = tqdm(unit='MB', unit_scale=True, desc='Download')
    last = [0]
    def hook(count, block, total):
        downloaded = count * block
        bar.total = total
        bar.update(downloaded - last[0])
        last[0] = downloaded
    urllib.request.urlretrieve(url, path, hook)
    bar.close()
    print(f'Saved ({os.path.getsize(path)/1e6:.0f} MB): {path}')
    return path

pgn_path = download_lichess(LICHESS_MONTH, DATA_DIR)

In [ ]:
import multiprocessing as mp

NUM_EXTRACT_WORKERS = max(1, os.cpu_count() - 2)  # leave 2 cores for main thread + OS
print(f'Extract workers : {NUM_EXTRACT_WORKERS}  (of {os.cpu_count()} CPUs)')
GAME_BATCH          = 600   # games per worker dispatch

# Sentinel returned by visitor for skipped (filtered-out) games
_SKIPPED = object()

class _FilterVisitor(chess.pgn.GameBuilder):
    """Skip movetext parsing for games that fail the ELO/TC filter.
    python-chess calls end_headers() before parsing moves; returning
    chess.pgn.SKIP tells it to discard the movetext and advance the stream."""
    _accepted = False

    def end_headers(self):
        h = self.game.headers
        try:
            base  = int(h.get('TimeControl', '0').split('+')[0])
            w_elo = int(h.get('WhiteElo', '0'))
            b_elo = int(h.get('BlackElo', '0'))
            result = h.get('Result', '*')
            if (base  >= MIN_TC_SECS and
                w_elo >= MIN_ELO and
                b_elo >= MIN_ELO and
                result in ('1-0', '0-1', '1/2-1/2')):
                self._accepted = True
                return super().end_headers()
        except ValueError:
            pass
        return chess.pgn.SKIP   # skip move parsing for this game

    def result(self):
        return super().result() if self._accepted else _SKIPPED


def _capture_frame(board):
    """14-plane snapshot — cheaper than board.copy() for history."""
    f = np.zeros((14, 8, 8), dtype=np.float32)
    for i, pt in enumerate(_PIECE_TYPES):
        for sq in board.pieces(pt, chess.WHITE):
            f[i,     sq // 8, sq % 8] = 1.0
        for sq in board.pieces(pt, chess.BLACK):
            f[6 + i, sq // 8, sq % 8] = 1.0
    if board.is_repetition(2): f[12] = 1.0
    if board.is_repetition(3): f[13] = 1.0
    return f


def _encode_game_batch(game_list):
    """Worker: encode positions for (moves_uci, result) tuples.
    Uses 14-plane frame history instead of Board copies — eliminates board.copy().
    SKIP_HALF / SAMPLE_EVERY / move_to_index inherited via fork."""
    s_out, p_out, v_out = [], [], []
    for moves_uci, result in game_list:
        board      = chess.Board()
        frame_hist = []   # list of (14,8,8) float32, most-recent first
        half       = 0
        for uci in moves_uci:
            move = chess.Move.from_uci(uci)
            half += 1
            if half > SKIP_HALF and half % SAMPLE_EVERY == 0:
                vc = (1 if result == '1/2-1/2' else
                      0 if ((result == '1-0'  and board.turn == chess.WHITE) or
                            (result == '0-1'  and board.turn == chess.BLACK))
                      else 2)
                # Build (120,8,8): fill current board inline, then paste pre-computed frames
                t = np.zeros((120, 8, 8), dtype=np.float32)
                for i, pt in enumerate(_PIECE_TYPES):
                    for sq in board.pieces(pt, chess.WHITE):
                        t[i,     sq // 8, sq % 8] = 1.0
                    for sq in board.pieces(pt, chess.BLACK):
                        t[6 + i, sq // 8, sq % 8] = 1.0
                if board.is_repetition(2): t[12] = 1.0
                if board.is_repetition(3): t[13] = 1.0
                for j, frm in enumerate(frame_hist[:7]):
                    t[(j + 1) * 14:(j + 2) * 14] = frm
                if board.has_kingside_castling_rights(chess.WHITE):  t[112] = 1.0
                if board.has_queenside_castling_rights(chess.WHITE): t[113] = 1.0
                if board.has_kingside_castling_rights(chess.BLACK):  t[114] = 1.0
                if board.has_queenside_castling_rights(chess.BLACK): t[115] = 1.0
                if board.ep_square is not None:
                    t[116, board.ep_square // 8, board.ep_square % 8] = 1.0
                if board.turn == chess.BLACK: t[117] = 1.0
                t[118] = board.fullmove_number / 512.0
                t[119] = board.halfmove_clock  / 100.0
                s_out.append(t)
                p_out.append(move_to_index(move))
                v_out.append(vc)
            # Capture 14-plane frame BEFORE push — correct repetition state, no Board.copy()
            frame_hist.insert(0, _capture_frame(board))
            if len(frame_hist) > 7:
                frame_hist.pop()
            board.push(move)
    return s_out, p_out, v_out
def extract_positions_chunked(pgn_zst_path, out_dir):
    prog_path = os.path.join(out_dir, 'extract_progress.json')
    if os.path.exists(prog_path):
        with open(prog_path) as f:
            prog = json.load(f)
        games_done     = prog['games_done']
        chunks_done    = prog['chunks_done']
        chunk_paths    = prog['chunk_paths']
        total_pos_done = prog.get('total_positions', 0)
        print(f'Resuming from game {games_done:,}  |  {chunks_done} chunks  |  {total_pos_done:,} positions saved')
    else:
        games_done = chunks_done = total_pos_done = 0
        chunk_paths = []

    def _save_progress():
        with open(prog_path, 'w') as f:
            json.dump({'games_done': games_done, 'chunks_done': chunks_done,
                       'chunk_paths': chunk_paths,
                       'total_positions': total_pos_done}, f)

    # Pre-allocate buffers once — filled in-place, flush via view (no copy, no RAM doubling).
    # Capacity = chunk size + max positions that can be in-flight when threshold is crossed.
    _IN_FLIGHT_CAP = (NUM_EXTRACT_WORKERS * 2 + 1) * GAME_BATCH * 100  # 100-move games = 64 pos
    _BUF_CAP       = CHUNK_POSITIONS + _IN_FLIGHT_CAP
    s_buf = np.empty((_BUF_CAP, 120, 8, 8), dtype=np.float32)
    p_buf = np.empty(_BUF_CAP,               dtype=np.int32)
    v_buf = np.empty(_BUF_CAP,               dtype=np.int8)
    n_buf = 0   # positions written so far in current chunk

    def _fill(s, p, v):
        """Copy worker results into pre-allocated buffers. Returns number of positions."""
        nonlocal n_buf
        m = len(v)
        if n_buf + m > _BUF_CAP:
            raise RuntimeError(f'Buffer overflow: n_buf={n_buf}+m={m} > cap={_BUF_CAP}. '
                               f'Increase _IN_FLIGHT_CAP multiplier.')
        s_buf[n_buf:n_buf + m] = np.asarray(s, dtype=np.float32)
        p_buf[n_buf:n_buf + m] = p
        v_buf[n_buf:n_buf + m] = v
        n_buf += m
        return m

    def _drain(pending, chunk_bar):
        i = 0
        while i < len(pending):
            if pending[i].ready():
                chunk_bar.update(_fill(*pending[i].get()))
                pending.pop(i)
            else:
                i += 1

    def _flush(chunk_bar):
        """Save current buffer slice to Drive. Zero-copy: savez reads the view."""
        nonlocal n_buf, total_pos_done, chunks_done
        chunk_bar.close()
        path = os.path.join(out_dir, f'raw_chunk_{chunks_done:03d}.npz')
        np.savez_compressed(path,
            states   = s_buf[:n_buf],
            policies = p_buf[:n_buf],
            values   = v_buf[:n_buf])
        chunk_paths.append(path)
        n_this   = n_buf
        total_pos_done += n_this
        chunks_done    += 1
        n_buf = 0          # reset pointer — reuse same allocation
        _save_progress()
        sz = os.path.getsize(path) / 1e9
        print(f'Chunk {chunks_done-1:03d}: {n_this:,} pos  {sz:.2f} GB  |  '
              f'total {total_pos_done:,}')
        gc.collect()
        return chunks_done >= MAX_CHUNKS

    games_seen     = 0
    games_accepted = 0
    pending        = []
    game_batch     = []

    mp_ctx = mp.get_context('fork')
    pool   = mp_ctx.Pool(NUM_EXTRACT_WORKERS)
    chunk_bar = tqdm(total=CHUNK_POSITIONS, desc=f'Chunk {chunks_done:03d}',
                     unit='pos', unit_scale=True)

    try:
        dctx = zstd.ZstdDecompressor()
        with open(pgn_zst_path, 'rb') as fh:
            with dctx.stream_reader(fh) as reader:
                text = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')

                # Fast-forward past already-accepted games
                if games_done > 0:
                    skip_bar = tqdm(total=games_done, desc='Fast-forwarding', unit='game')
                    accepted_so_far = 0
                    while accepted_so_far < games_done:
                        g = chess.pgn.read_game(text, Visitor=_FilterVisitor)
                        if g is None: break
                        if g is not _SKIPPED:
                            accepted_so_far += 1
                            skip_bar.update(1)
                    skip_bar.close()

                stop = False
                while not stop:
                    game = chess.pgn.read_game(text, Visitor=_FilterVisitor)
                    if game is None:
                        break

                    games_seen += 1
                    if game is _SKIPPED:
                        continue

                    result    = game.headers.get('Result')
                    moves_uci = [m.uci() for m in game.mainline_moves()]
                    game_batch.append((moves_uci, result))
                    games_done     += 1
                    games_accepted += 1

                    if len(game_batch) >= GAME_BATCH:
                        pending.append(pool.apply_async(_encode_game_batch, (game_batch,)))
                        game_batch = []
                        # Backpressure: block on oldest when queue is too deep
                        while len(pending) > NUM_EXTRACT_WORKERS * 2:
                            chunk_bar.update(_fill(*pending[0].get()))
                            pending.pop(0)

                    _drain(pending, chunk_bar)

                    skip_pct = (games_seen - games_accepted) / max(games_seen, 1) * 100
                    chunk_bar.set_postfix({
                        'total': f'{total_pos_done + n_buf:,}',
                        'games': f'{games_accepted:,}',
                        'skip%': f'{skip_pct:.0f}%',
                        'queue': len(pending),
                    })

                    if n_buf >= CHUNK_POSITIONS:
                        for fut in pending:
                            chunk_bar.update(_fill(*fut.get()))
                        pending.clear()
                        stop = _flush(chunk_bar)
                        if not stop:
                            chunk_bar = tqdm(total=CHUNK_POSITIONS,
                                             desc=f'Chunk {chunks_done:03d}',
                                             unit='pos', unit_scale=True)

        # Drain any remaining work and flush trailing partial chunk
        if not stop:
            if game_batch:
                pending.append(pool.apply_async(_encode_game_batch, (game_batch,)))
            for fut in pending:
                chunk_bar.update(_fill(*fut.get()))
            chunk_bar.close()
            if n_buf > 0:
                _flush(chunk_bar)  # ignore return — we're done anyway

    finally:
        del s_buf, p_buf, v_buf
        pool.close()
        pool.join()

    print(f'\nDone: {games_accepted:,} accepted  |  {chunks_done} chunks  |  {total_pos_done:,} total positions')
    return chunk_paths

gc.collect()
chunk_paths = extract_positions_chunked(pgn_path, DATA_DIR)
print(f'Chunks available: {len(chunk_paths)}')

In [ ]:

# ── Storage cleanup — run once after any interrupted rechunk ──────────────────
# Scans disk to find valid-but-unjournalled chunks, deletes partial files from
# the interrupted chunk, then frees all original raw_chunks that are fully done.
import glob as _glob

_RECHUNK_CACHE = os.path.join(DATA_DIR, 'rechunk_done.json')
_minis_per     = math.ceil(CHUNK_POSITIONS / MINI_CHUNK_SIZE)  # = 3 full mini-chunks per original

if os.path.exists(_RECHUNK_CACHE):
    with open(_RECHUNK_CACHE) as f:
        _rc = json.load(f)
    _n_done = len(_rc.get('chunks_mini_counts', []))
else:
    _n_done = 0

# Walk forward from n_done: count chunks whose mini-files are all present on disk
# (these are valid but not recorded in JSON because the interrupt hit before json.dump)
_confirmed = _n_done
if 'chunk_paths' in vars():
    _total_orig = len(chunk_paths)
else:
    _total_orig = len(sorted(_glob.glob(os.path.join(DATA_DIR, 'raw_chunk_*.npz'))))

for _i in range(_n_done, _total_orig):
    _existing = sorted(_glob.glob(os.path.join(DATA_DIR, f'mini_chunk_{_i:03d}_*.npz')))
    if len(_existing) == _minis_per:
        _confirmed += 1   # complete set — valid even though not in JSON
    elif _existing:
        # Partial write — this is the corrupt chunk; delete its files
        for _f in _existing:
            os.remove(_f)
            print(f'Deleted partial  : {os.path.basename(_f)}')
        break
    else:
        break   # no files at all — clean from here

print(f'Confirmed complete: chunks 0–{_confirmed - 1}  ({_confirmed} total, '
      f'{_n_done} in JSON + {_confirmed - _n_done} valid-but-unjournalled)')

# Delete original raw_chunks for all confirmed-complete chunks
_freed = 0
for _i in range(_confirmed):
    _rcp = os.path.join(DATA_DIR, f'raw_chunk_{_i:03d}.npz')
    if os.path.exists(_rcp):
        _sz = os.path.getsize(_rcp)
        os.remove(_rcp)
        _freed += _sz
        print(f'Deleted original : raw_chunk_{_i:03d}.npz  ({_sz/1e9:.2f} GB)')

print(f'\nFreed : {_freed/1e9:.1f} GB')
print('Rerun the rechunk cell to continue from where it left off.')


In [ ]:
import glob as _glob

# Reconstruct chunk_paths in case extraction was skipped this session
if 'chunk_paths' not in vars():
    _prog = os.path.join(DATA_DIR, 'extract_progress.json')
    if os.path.exists(_prog):
        with open(_prog) as _f: chunk_paths = json.load(_f)['chunk_paths']
    else:
        chunk_paths = sorted(_glob.glob(os.path.join(DATA_DIR, 'raw_chunk_*.npz')))
    print(f'Reconstructed chunk_paths: {len(chunk_paths)} chunks')

RECHUNK_CACHE = os.path.join(DATA_DIR, 'rechunk_done.json')

# Load any existing progress
if os.path.exists(RECHUNK_CACHE):
    with open(RECHUNK_CACHE) as f:
        _rc = json.load(f)
    mini_chunk_paths    = _rc['mini_chunk_paths']
    _chunks_mini_counts = _rc.get('chunks_mini_counts', [])
else:
    mini_chunk_paths    = []
    _chunks_mini_counts = []

_n_done = len(_chunks_mini_counts)

# Short-circuit only when every original chunk is journalled
if _n_done >= len(chunk_paths):
    print(f'Rechunking already done: {len(mini_chunk_paths)} mini-chunks')
    print(f'  Mini-chunk size: {MINI_CHUNK_SIZE:,} positions  '
          f'({MINI_CHUNK_SIZE * 120 * 8 * 8 * 4 / 1e9:.1f} GB decompressed)')
else:
    _minis_per = math.ceil(CHUNK_POSITIONS / MINI_CHUNK_SIZE)
    if _n_done == 0:
        _est_hrs = len(chunk_paths) * 5 / 60
        print(f'Rechunking {len(chunk_paths)} original chunks → ~{len(chunk_paths) * _minis_per} mini-chunks')
        print(f'Each mini-chunk: {MINI_CHUNK_SIZE:,} positions  '
              f'({MINI_CHUNK_SIZE * 120 * 8 * 8 * 4 / 1e9:.1f} GB decompressed)')
        print(f'Estimated time : ~5 min/chunk × {len(chunk_paths)} ≈ {_est_hrs:.1f} hrs  '
              f'(run on CPU-only Colab to save A100 compute units)')
    else:
        print(f'Resuming rechunk: {_n_done}/{len(chunk_paths)} chunks in JSON, '
              f'{len(mini_chunk_paths)} mini-chunks so far')
    print()

    for i, cp in enumerate(tqdm(chunk_paths, desc='Rechunking', unit='chunk')):
        prefix   = f'mini_chunk_{i:03d}'
        existing = sorted(_glob.glob(os.path.join(DATA_DIR, f'{prefix}_*.npz')))
        if existing:
            # Files exist on disk — journal them if not already recorded
            if i >= _n_done:
                mini_chunk_paths.extend(existing)
                _chunks_mini_counts.append(len(existing))
                with open(RECHUNK_CACHE, 'w') as f:
                    json.dump({'mini_chunk_paths':   mini_chunk_paths,
                               'mini_chunk_size':    MINI_CHUNK_SIZE,
                               'chunks_mini_counts': _chunks_mini_counts}, f)
            if os.path.exists(cp):
                os.remove(cp)   # free original — mini-chunks already on disk
            continue

        raw   = np.load(cp)
        s_all = raw['states']
        p_all = raw['policies']
        v_all = raw['values']
        del raw

        n = len(v_all); minis = []
        for j, start in enumerate(range(0, n, MINI_CHUNK_SIZE)):
            end = min(start + MINI_CHUNK_SIZE, n)
            out = os.path.join(DATA_DIR, f'{prefix}_{j:02d}.npz')
            np.savez_compressed(out,
                states   = s_all[start:end],
                policies = p_all[start:end],
                values   = v_all[start:end])
            minis.append(out)

        del s_all, p_all, v_all; gc.collect()
        mini_chunk_paths.extend(minis)
        _chunks_mini_counts.append(len(minis))

        with open(RECHUNK_CACHE, 'w') as f:
            json.dump({'mini_chunk_paths':   mini_chunk_paths,
                       'mini_chunk_size':    MINI_CHUNK_SIZE,
                       'chunks_mini_counts': _chunks_mini_counts}, f)

        os.remove(cp)   # free original after mini-chunks written + JSON updated

    print(f'Done: {len(mini_chunk_paths)} mini-chunks from {len(chunk_paths)} original chunks')


In [ ]:
import glob
if 'chunk_paths' not in vars():
    _prog = os.path.join(DATA_DIR, 'extract_progress.json')
    if os.path.exists(_prog):
        with open(_prog) as _f:
            chunk_paths = json.load(_f)['chunk_paths']
    else:
        chunk_paths = sorted(glob.glob(os.path.join(DATA_DIR, 'raw_chunk_*.npz')))
    print(f'Reconstructed chunk_paths: {len(chunk_paths)} chunks')

RECHUNK_CACHE  = os.path.join(DATA_DIR, 'rechunk_done.json')
RESAMPLE_CACHE = os.path.join(DATA_DIR, 'resample_done.json')

if os.path.exists(RECHUNK_CACHE):
    with open(RECHUNK_CACHE) as f:
        _rc = json.load(f)
    _all                = _rc['mini_chunk_paths']
    _chunks_mini_counts = _rc['chunks_mini_counts']
    n_val_orig  = max(1, round(len(chunk_paths) * VAL_SPLIT))
    n_val_mini  = sum(_chunks_mini_counts[-n_val_orig:])
    val_chunks   = _all[-n_val_mini:]
    train_chunks = _all[:-n_val_mini]
    with open(os.path.join(DATA_DIR, 'extract_progress.json')) as _f:
        N_total = json.load(_f)['total_positions']
    # Persist the mini-chunk split so resample_done.json always reflects current data
    with open(RESAMPLE_CACHE, 'w') as f:
        json.dump({'train_chunks': train_chunks, 'val_chunks': val_chunks,
                   'N_total': N_total, 'source': 'rechunk'}, f)
    print(f'Split  (mini-chunks)  {len(train_chunks)} train  |  {len(val_chunks)} val')
    print(f'Total positions: {N_total:,}')

elif os.path.exists(RESAMPLE_CACHE):
    with open(RESAMPLE_CACHE) as f:
        rs = json.load(f)
    train_chunks = rs['train_chunks']
    val_chunks   = rs['val_chunks']
    N_total      = rs['N_total']
    _src         = rs.get('source', 'original')
    print(f'Split cached ({_src} chunks): {N_total:,} positions')
    print(f'  {len(train_chunks)} train chunks  |  {len(val_chunks)} val chunks')

else:
    n_val = max(1, round(len(chunk_paths) * VAL_SPLIT))
    val_chunks   = chunk_paths[-n_val:]
    train_chunks = chunk_paths[:-n_val]
    _prog = os.path.join(DATA_DIR, 'extract_progress.json')
    with open(_prog) as _f:
        N_total = json.load(_f)['total_positions']
    with open(RESAMPLE_CACHE, 'w') as f:
        json.dump({'train_chunks': train_chunks, 'val_chunks': val_chunks,
                   'N_total': N_total, 'source': 'original'}, f)
    print(f'Total: {N_total:,}  |  {len(train_chunks)} train chunks  |  {len(val_chunks)} val chunks')
    gc.collect()


In [ ]:
print('Aggregating dataset stats from chunks...')
wdl_agg    = np.zeros(3, dtype=np.int64)
pol_counts = np.zeros(4672, dtype=np.int64)
N_viz      = 0

for cp in tqdm(train_chunks + val_chunks, desc='Scanning', unit='chunk'):
    with np.load(cp) as d:
        v = d['values'].astype(np.int8)
        p = d['policies'].astype(np.int64)
    for k in range(3):
        wdl_agg[k] += int((v == k).sum())
    np.add.at(pol_counts, p, 1)
    N_viz += len(v)

wdl = wdl_agg
N   = N_viz

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 1. WDL distribution
wdl_labels = ['Win (0)', 'Draw (1)', 'Loss (2)']
wdl_colors = ['#2ecc71', '#95a5a6', '#e74c3c']
bars = axes[0].bar(wdl_labels, wdl, color=wdl_colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Value (WDL) Distribution', fontsize=12)
axes[0].set_ylabel('Positions')
for bar, v in zip(bars, wdl):
    axes[0].text(bar.get_x() + bar.get_width()/2, v + N*0.005,
                 f'{v/N*100:.1f}%', ha='center', fontsize=10)

# 2. Policy index distribution
axes[1].hist(np.where(pol_counts > 0)[0], bins=100,
             weights=pol_counts[pol_counts > 0],
             color='#2980b9', edgecolor='none', alpha=0.8)
axes[1].set_title('Policy Index Distribution (move diversity)', fontsize=12)
axes[1].set_xlabel('Move index (0–4671)')
axes[1].set_ylabel('Frequency')
axes[1].axvline(4672/2, color='red', lw=1, ls='--', label='midpoint')
axes[1].legend(fontsize=9)

plt.suptitle(f'Dataset: {N:,} positions  |  {LICHESS_MONTH}  |  1500+ ELO  |  10-min+ TC',
             fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'dataset_stats.png'), dpi=150)
plt.show()

top10 = np.argsort(pol_counts)[-10:][::-1]
print(f'\nTotal positions : {N:,}')
print(f'Train chunks    : {len(train_chunks)}  |  Val chunks: {len(val_chunks)}')
print('\nTop-10 most frequent policy indices:')
for rank, idx in enumerate(top10, 1):
    src_sq = chess.square_name(idx // 73)
    print(f'  {rank:2}. index {idx:4d}  (src={src_sq}, plane={idx%73})  '
          f'count={pol_counts[idx]:,}  ({pol_counts[idx]/N*100:.2f}%)')


In [ ]:
# Derive steps from actual position count, not assumed chunk sizes.
# Per-chunk rechunking creates partial mini-chunks (402,848 pos instead of 1,048,576)
# at the end of every original chunk — computing from N_total avoids overcounting.
train_frac      = len(train_chunks) / (len(train_chunks) + len(val_chunks))
train_total_pos = round(N_total * train_frac)
steps_per_epoch = math.ceil(train_total_pos / BATCH_SIZE)

print(f'Train chunks    : {len(train_chunks)}  |  Val chunks: {len(val_chunks)}')
print(f'Train positions : ~{train_total_pos:,}  ({train_frac*100:.1f}% of {N_total:,})')
print(f'Steps per epoch : {steps_per_epoch:,}  (batch {BATCH_SIZE:,})')


## Training
- **Policy loss** — CrossEntropy(logits, move_index): learn which move to play
- **Value loss** — CrossEntropy(logits, wdl_class): learn game outcome (W=0, D=1, L=2)
- Mixed-precision (AMP) + gradient clipping + cosine LR decay

In [ ]:
import threading, random
from queue import Queue as _TQueue

optimizer    = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps  = EPOCHS * steps_per_epoch
warmup_steps = steps_per_epoch

warmup_sched = LinearLR(optimizer, start_factor=0.01, end_factor=1.0,
                         total_iters=warmup_steps)
cosine_sched = CosineAnnealingLR(optimizer, T_max=total_steps - warmup_steps,
                                  eta_min=1e-5)
scheduler    = SequentialLR(optimizer,
                             schedulers=[warmup_sched, cosine_sched],
                             milestones=[warmup_steps])
amp_device   = 'cuda' if torch.cuda.is_available() else 'cpu'
scaler       = torch.amp.GradScaler(amp_device, enabled=False)  # bfloat16 has float32 exponent range — no overflow, scaling is a no-op

log = {'train_p': [], 'train_v': [], 'val_p': [], 'val_v': [],
       'train_acc': [], 'val_acc': [], 'grad_norm': [], 'lr': []}
best_val      = float('inf')
no_improve    = 0
stopped_epoch = EPOCHS
start_epoch   = 1

# ── Resume ───────────────────────────────────────────────────────────────────
if os.path.exists(RESUME_PATH):
    print(f'Resuming from: {RESUME_PATH}')
    ckpt = torch.load(RESUME_PATH, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    scaler.load_state_dict(ckpt['scaler_state_dict'])
    log         = ckpt['log']
    best_val    = ckpt['best_val']
    no_improve  = ckpt['no_improve']
    start_epoch = ckpt['epoch'] + 1
    print(f'Resumed at epoch {start_epoch}  |  best_val={best_val:.4f}'
          f'  |  no_improve={no_improve}/{EARLY_STOP_PAT}')
    if start_epoch > EPOCHS:
        print('Training already complete.')
else:
    print('No checkpoint — starting fresh.')

baseline_p = math.log(4672)
baseline_v = math.log(3)

print()
print('=' * 90)
print(f'  Training  |  epochs {start_epoch}–{EPOCHS}  |  patience={EARLY_STOP_PAT}'
      f'  |  batch={BATCH_SIZE:,}  |  lr={LR}')
print(f'  {steps_per_epoch:,} steps/epoch  |  {len(train_chunks)} train chunks  |  '
      f'{len(val_chunks)} val chunks  |  {N_WORKERS} workers')
print(f'  Baselines  policy={baseline_p:.3f}  value={baseline_v:.3f}'
      f'  acc≈{1/4672*100:.3f}% (random)')
print('=' * 90)
print(f'{"Ep":>5}  {"Tr-P":>7} {"Tr-V":>7} {"Va-P":>7} {"Va-V":>7}'
      f'  {"Tr-Acc":>7} {"Va-Acc":>7}  {"GNorm":>6}  {"LR":>9}  {"Time":>6}')
print('-' * 90)

# batch_q maxsize = MINI_CHUNK_SIZE // BATCH_SIZE so one chunk fills the queue
# exactly once — alignment keeps workers lock-free and RAM flat regardless of batch size.
_QUEUE_MAXSIZE = MINI_CHUNK_SIZE // BATCH_SIZE   # 32 at batch=32,768 / 16 at batch=65,536

def run_epoch_parallel(chunk_files, train=True):
    """N_WORKERS threads decompress mini-chunks in parallel, feeding a shared batch queue.
    Alignment: MINI_CHUNK_SIZE = BATCH_SIZE × _QUEUE_MAXSIZE → worker fills queue
    exactly once per chunk, never blocks on put, immediately starts next decompression."""
    model.train(train)
    p_sum = v_sum = correct = total_samples = 0
    gnorms = []

    work_q = _TQueue()
    for path in chunk_files:
        work_q.put(path)
    for _ in range(N_WORKERS):
        work_q.put(None)

    batch_q = _TQueue(maxsize=_QUEUE_MAXSIZE)
    exc_box = [None]
    active  = [N_WORKERS]
    lock    = threading.Lock()

    def _worker():
        try:
            while True:
                path = work_q.get()
                if path is None:
                    break
                raw   = np.load(path)
                s_all = raw['states']
                p_all = raw['policies'].astype(np.int64)
                v_all = raw['values'].astype(np.int64)
                del raw
                n   = len(v_all)
                idx = np.random.permutation(n) if train else np.arange(n)
                for s in range(0, n, BATCH_SIZE):
                    b  = idx[s:s + BATCH_SIZE]
                    ts = torch.from_numpy(s_all[b].copy())
                    if device.type == 'cuda':
                        ts = ts.pin_memory()
                    batch_q.put((ts,
                                 torch.from_numpy(p_all[b].copy()),
                                 torch.from_numpy(v_all[b].copy())))
                del s_all, p_all, v_all
                # no gc.collect() — del with refcount=0 frees immediately; gc only needed for cycles
        except Exception as e:
            exc_box[0] = e
        finally:
            with lock:
                active[0] -= 1
                if active[0] == 0:
                    batch_q.put(None)

    for _ in range(N_WORKERS):
        threading.Thread(target=_worker, daemon=True).start()

    # Use steps_per_epoch for train, scale proportionally for val
    n_approx = steps_per_epoch if train else max(1, round(steps_per_epoch * len(chunk_files) / max(len(train_chunks), 1)))

    ctx     = torch.enable_grad() if train else torch.no_grad()
    desc    = '  train' if train else '    val'
    _step   = 0
    _t_step = time.time()

    with ctx, tqdm(total=n_approx, desc=desc, unit='step', leave=False) as pbar:
        while True:
            item = batch_q.get()
            if item is None:
                if exc_box[0]: raise exc_box[0]
                break
            if exc_box[0]: raise exc_box[0]

            states, pol_tgt, val_tgt = item
            states  = states.to(device,  non_blocking=True)
            pol_tgt = pol_tgt.to(device, non_blocking=True)
            val_tgt = val_tgt.to(device, non_blocking=True)

            if train:
                optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(amp_device, dtype=torch.bfloat16):
                pol_out, val_out = model(states)
                p_loss = F.cross_entropy(pol_out, pol_tgt)
                v_loss = F.cross_entropy(val_out, val_tgt)
            loss = p_loss + v_loss
            if train:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                gnorms.append(gnorm.item())
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()

            bs       = pol_tgt.size(0)
            correct += (pol_out.argmax(dim=1) == pol_tgt).sum().item()
            total_samples += bs
            p_sum += p_loss.item() * bs
            v_sum += v_loss.item() * bs
            _step += 1
            pbar.update(1)
            if _step % 50 == 0:
                _now    = time.time()
                _ms     = (_now - _t_step) * 1000 / 50
                _t_step = _now
                _ram    = psutil.Process().memory_info().rss / 1e9
                _vram   = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
                pbar.set_postfix(RAM=f'{_ram:.0f}G', VRAM=f'{_vram:.1f}G',
                                 ms=f'{_ms:.0f}', refresh=False)

    acc       = correct / total_samples * 100
    avg_gnorm = float(np.mean(gnorms)) if gnorms else 0.0
    return p_sum / total_samples, v_sum / total_samples, acc, avg_gnorm


for epoch in range(start_epoch, EPOCHS + 1):
    shuffled_train = list(train_chunks)
    random.shuffle(shuffled_train)

    t0 = time.time()
    tp, tv, t_acc, gnorm = run_epoch_parallel(shuffled_train, train=True)
    vp, vv, v_acc, _     = run_epoch_parallel(val_chunks,     train=False)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    elapsed = time.time() - t0

    lr_now    = scheduler.get_last_lr()[0]
    ram_gb    = psutil.Process().memory_info().rss / 1e9
    vram_gb   = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
    val_total = vp + vv

    log['train_p'].append(tp);      log['train_v'].append(tv)
    log['val_p'].append(vp);        log['val_v'].append(vv)
    log['train_acc'].append(t_acc); log['val_acc'].append(v_acc)
    log['grad_norm'].append(gnorm); log['lr'].append(lr_now)

    if val_total < best_val:
        best_val   = val_total
        no_improve = 0
        torch.save({'model_state_dict': model.state_dict(),
                    'epoch': epoch, 'val_policy_loss': vp,
                    'val_value_loss': vv}, MODEL_PATH)
        tag = '  ✓'
    else:
        no_improve += 1
        tag = f'  -{no_improve}'

    torch.save({'model_state_dict':     model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'scaler_state_dict':    scaler.state_dict(),
                'epoch': epoch, 'log': log,
                'best_val': best_val, 'no_improve': no_improve,
                'val_policy_loss': vp, 'val_value_loss': vv}, RESUME_PATH)

    ms_step = elapsed * 1000 / max(steps_per_epoch, 1)
    print(f'{epoch:>5}/{EPOCHS}  {tp:>7.4f} {tv:>7.4f} {vp:>7.4f} {vv:>7.4f}'
          f'  {t_acc:>6.2f}% {v_acc:>6.2f}%  {gnorm:>6.3f}  {lr_now:>9.2e}'
          f'  {elapsed:>4.0f}s{tag}')
    print(f'         RAM={ram_gb:.1f}GB  VRAM={vram_gb:.1f}GB  ~{ms_step:.0f}ms/step')

    if no_improve >= EARLY_STOP_PAT:
        stopped_epoch = epoch
        print(f'\nEarly stop — no improvement for {EARLY_STOP_PAT} epochs.')
        break

print('-' * 90)
print(f'Stopped : epoch {stopped_epoch}/{EPOCHS}  |  best val loss {best_val:.4f}')
print(f'Policy  {baseline_p:.3f} → {min(log["val_p"]):.4f}'
      f'  ({(1 - min(log["val_p"])/baseline_p)*100:.1f}% below random)')
print(f'Value   {baseline_v:.3f} → {min(log["val_v"]):.4f}'
      f'  ({(1 - min(log["val_v"])/baseline_v)*100:.1f}% below random)')
print(f'Top-1 accuracy : {max(log["val_acc"]):.2f}%'
      f'  (random baseline {1/4672*100:.3f}%)')


In [ ]:
epochs_x  = list(range(1, len(log['val_p']) + 1))
n_epochs  = len(epochs_x)
best_ep   = int(np.argmin([p + v for p, v in zip(log['val_p'], log['val_v'])])) + 1

C_TRAIN, C_VAL, C_RAND = '#1d4ed8', '#b91c1c', '#9ca3af'
C_LR,    C_GN          = '#15803d', '#7c3aed'

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
plt.rcParams.update({'font.size': 11, 'axes.titlesize': 12})

def _annotate_best(ax, y_vals, color, is_max=False, fmt='{:.4f}'):
    """Mark best epoch with vertical line + value label."""
    best_i = int(np.argmax(y_vals) if is_max else np.argmin(y_vals))
    ax.axvline(epochs_x[best_i], color=color, lw=1.5, ls='--', alpha=0.45)
    ax.annotate(fmt.format(y_vals[best_i]),
                xy=(epochs_x[best_i], y_vals[best_i]),
                xytext=(7, 7), textcoords='offset points',
                fontsize=9.5, color=color, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.2', fc='white', ec=color, alpha=0.85))

# ── 1. Policy loss ────────────────────────────────────────────────────────────
ax = axes[0, 0]
ax.fill_between(epochs_x, log['train_p'], log['val_p'], alpha=0.07, color=C_VAL)
ax.plot(epochs_x, log['train_p'], 'o-', color=C_TRAIN, lw=2, ms=4, label='Train')
ax.plot(epochs_x, log['val_p'],   'o-', color=C_VAL,   lw=2, ms=4, label='Val')
ax.axhline(math.log(4672), color=C_RAND, lw=1.2, ls='--',
           label=f'Random ({math.log(4672):.3f})')
_annotate_best(ax, log['val_p'], C_VAL)
ax.set_title('Policy Loss  (cross-entropy)', fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.legend(framealpha=0.9); ax.grid(True, alpha=0.3)
ax.set_xticks(epochs_x)

# ── 2. Value loss ─────────────────────────────────────────────────────────────
ax = axes[0, 1]
ax.fill_between(epochs_x, log['train_v'], log['val_v'], alpha=0.07, color=C_VAL)
ax.plot(epochs_x, log['train_v'], 'o-', color=C_TRAIN, lw=2, ms=4, label='Train')
ax.plot(epochs_x, log['val_v'],   'o-', color=C_VAL,   lw=2, ms=4, label='Val')
ax.axhline(math.log(3), color=C_RAND, lw=1.2, ls='--',
           label=f'Random ({math.log(3):.3f})')
_annotate_best(ax, log['val_v'], C_VAL)
ax.set_title('Value Loss — WDL  (cross-entropy)', fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.legend(framealpha=0.9); ax.grid(True, alpha=0.3)
ax.set_xticks(epochs_x)

# ── 3. Policy top-1 accuracy ──────────────────────────────────────────────────
ax = axes[1, 0]
ax.fill_between(epochs_x, log['train_acc'], log['val_acc'], alpha=0.07, color=C_TRAIN)
ax.plot(epochs_x, log['train_acc'], 'o-', color=C_TRAIN, lw=2, ms=4, label='Train')
ax.plot(epochs_x, log['val_acc'],   'o-', color=C_VAL,   lw=2, ms=4, label='Val')
ax.axhline(1/4672*100, color=C_RAND, lw=1.2, ls='--',
           label=f'Random ({1/4672*100:.3f}%)')
_annotate_best(ax, log['val_acc'], C_VAL, is_max=True, fmt='{:.2f}%')
ax.set_title('Policy Top-1 Accuracy  (% matching human move)', fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy (%)')
ax.legend(framealpha=0.9); ax.grid(True, alpha=0.3)
ax.set_xticks(epochs_x)

# ── 4. LR schedule + gradient norm ───────────────────────────────────────────
ax_lr = axes[1, 1]
ax_gn = ax_lr.twinx()
l1, = ax_lr.plot(epochs_x, log['lr'],        'o-',  color=C_LR, lw=2,   ms=4, label='LR')
l2, = ax_gn.plot(epochs_x, log['grad_norm'], 's--', color=C_GN, lw=1.5, ms=4,
                 label='Grad norm', alpha=0.85)
ax_lr.axhline(1e-5, color=C_LR, lw=0.7, ls=':', alpha=0.4)
ax_lr.set_title('LR Schedule & Gradient Norm', fontweight='bold')
ax_lr.set_xlabel('Epoch')
ax_lr.set_ylabel('Learning Rate', color=C_LR)
ax_gn.set_ylabel('Grad Norm  (clip=1.0)', color=C_GN)
ax_lr.tick_params(axis='y', colors=C_LR)
ax_gn.tick_params(axis='y', colors=C_GN)
ax_lr.legend(handles=[l1, l2], loc='upper right', framealpha=0.9)
ax_lr.grid(True, alpha=0.3)
ax_lr.set_xticks(epochs_x)

# ── Title with key improvement metrics ───────────────────────────────────────
p_drop = (1 - min(log['val_p']) / math.log(4672)) * 100
v_drop = (1 - min(log['val_v']) / math.log(3))    * 100
plt.suptitle(
    f'ChessCNN Pretraining  ·  {LICHESS_MONTH}  ·  1500+ ELO  ·  10-min+ TC\n'
    f'{N_total:,} positions  ·  {n_epochs} epoch{"s" if n_epochs != 1 else ""}  ·  '
    f'Policy ↓{p_drop:.1f}%  ·  Value ↓{v_drop:.1f}%  (vs random baseline)',
    fontsize=12, fontweight='bold', y=1.01
)
plt.tight_layout()
curve_path = os.path.join(BASE_DIR, 'training_curves.png')
plt.savefig(curve_path, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {curve_path}')


## Save & Download
Download the model, then upload it to GCP as `chess_ai/game_engine/model/best_model.pth`.

In [ ]:
size_mb = os.path.getsize(MODEL_PATH) / 1e6
print(f'Model saved : {MODEL_PATH}  ({size_mb:.0f} MB)')

if USE_DRIVE:
    print('Google Drive is mounted — model is already in your Drive.')
    print(f'Access it any time at: MyDrive/chess_pretrain/pretrained_model.pth')
    print()
    print('To download now, run the cell below.')
else:
    print('Drive not mounted — download now before the session ends.')
    print()
    from google.colab import files as _cf
    _cf.download(MODEL_PATH)

print()
print('Next steps:')
print('  1. Upload pretrained_model.pth to your GCP instance')
print('  2. cp pretrained_model.pth chess_ai/game_engine/model/best_model.pth')
print('  3. cd chess_ai && python game_engine/main.py')

In [ ]:
# ── Manual download — run this cell whenever you're ready ────────────────────
from google.colab import files as _cf
_cf.download(MODEL_PATH)
print('Download triggered.')